In [1]:
"""
CAFA 6 Multi-Channel Protein Function Prediction Model
=======================================================

This script builds a multi-channel ensemble model combining:
1. CNN predictions (pre-computed)
2. ESM2 + MLP model
3. PPI neighbor-based scoring
4. Taxonomy-based priors
5. GO graph propagation

Final output: /kaggle/working/submission.tsv
"""

import os
import time
import pickle
import gc
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths - Competition Data
CAFA_TRAIN_TERMS = '/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv'
CAFA_TRAIN_TAXONOMY = '/kaggle/input/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv'
CAFA_TEST_TAXONOMY = '/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset-taxon-list.tsv'
GO_OBO_PATH = '/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo'
IA_TSV_PATH = '/kaggle/input/cafa-6-protein-function-prediction/Train/IA.tsv'

# Paths - ESM2 Embeddings
ESM2_EMBEDDINGS_PATH = '/kaggle/input/cafa6-protein-embeddings-esm2/protein_embeddings.npy'
ESM2_IDS_PATH        = '/kaggle/input/cafa6-protein-embeddings-esm2/protein_ids.csv'

# Paths - PPI Data
PPI_NEIGHBORS_PATH = '/kaggle/input/data-ppi/ppi_neighbors.pkl'

# Paths - GOA Data
GOA_CSV_PATH = '/kaggle/input/protein-go-annotations/goa_uniprot_all.csv'
GOA_TAXONOMY_CSV_PATH = '/kaggle/input/protein-go-annotations-taxonomy/protein_go_annotations.csv'

# Paths - Pre-computed CNN Submission
CNN_SUBMISSION_PATH = '/kaggle/input/notebookb401f587d7/submission.tsv'

# Output Path
OUTPUT_SUBMISSION_PATH = '/kaggle/working/submission.tsv'

# Hyperparameters
MIN_GO_FREQ = 30           # Minimum frequency for GO terms
ESM2_HIDDEN_DIM = 1024     # Hidden layer size for ESM2 MLP
ESM2_DROPOUT = 0.3         # Dropout for ESM2 MLP
ESM2_BATCH_SIZE = 64       # Batch size for ESM2 training
ESM2_NUM_EPOCHS = 8        # Number of epochs for ESM2 training
ESM2_LEARNING_RATE = 1e-4  # Learning rate for ESM2 MLP
ESM2_VAL_SPLIT = 0.1       # Validation split ratio

# Blending Weights
WEIGHT_CNN  = 0.30          # Weight for CNN channel
WEIGHT_ESM2 = 0.50          # Weight for ESM2 channel
WEIGHT_PPI  = 0.15          # Weight for PPI channel
WEIGHT_TAXO = 0.05          # Weight for Taxonomy channel

# Taxonomy Prior Smoothing
TAXO_SMOOTHING = 0.1       # Add-k smoothing
TAXO_MIN_PROB = 1e-4       # Minimum probability
TAXO_MAX_PROB = 0.99       # Maximum probability

# Submission Settings
TOP_K_PER_PROTEIN = 800    # Top K predictions per protein
MIN_SCORE_EPSILON = 1e-5   # Minimum score to avoid zeros

# GO propagation flag (keep True for now)
APPLY_GO_PROPAGATION = True

# ============================================================================
# DEVICE AND SEEDS
# ============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}", flush=True)

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def normalize_protein_id(raw_id: str) -> str:
    """
    Normalize FASTA and TSV IDs to UniProt-like accessions.
    
    Examples:
    - 'sp|P9WHI7|RECN_MYCT' -> 'P9WHI7'
    - 'tr|A0A009IHW8|SOME_NAME' -> 'A0A009IHW8'
    - 'Q9CQV8|10090' -> 'Q9CQV8'
    - 'Q9CQV8' -> 'Q9CQV8'
    """
    raw_id = str(raw_id).strip()
    if "|" in raw_id:
        parts = raw_id.split("|")
        if parts[0] in ("sp", "tr") and len(parts) >= 2:
            return parts[1]
        return parts[0]
    return raw_id


def format_time(seconds):
    """Format seconds into a readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}min"
    else:
        return f"{seconds/3600:.1f}h"


# ============================================================================
# DATA LOADERS
# ============================================================================

def load_cafa_train_labels(tsv_path):
    """
    Load CAFA training labels from train_terms.tsv.
    Returns: dict protein_id -> list of GO terms
    """
    protein_go_map = defaultdict(list)
    
    if not os.path.exists(tsv_path):
        print(f"  Warning: {tsv_path} not found", flush=True)
        return dict(protein_go_map)
    
    with open(tsv_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            raw_id = parts[0]
            
            # Skip header rows
            if raw_id.lower() in ("entryid", "protein_id", "entry_id"):
                continue
            
            protein_id = normalize_protein_id(raw_id)
            go_terms = parts[1:]
            for go in go_terms:
                if go and go.startswith("GO:"):
                    protein_go_map[protein_id].append(go)
    
    return dict(protein_go_map)


def load_goa_annotations(csv_path):
    """
    Load GOA annotations from CSV file.
    Returns: dict protein_id -> list of GO terms
    """
    protein_go_map = defaultdict(list)
    
    if not os.path.exists(csv_path):
        print(f"  Warning: {csv_path} not found", flush=True)
        return dict(protein_go_map)
    
    # Read CSV in chunks to save memory
    try:
        for chunk in pd.read_csv(csv_path, chunksize=100000, dtype=str):
            # Detect column names
            cols = chunk.columns.tolist()
            
            # Find protein ID column
            protein_col = None
            for c in cols:
                if 'protein' in c.lower() or 'entry' in c.lower() or 'accession' in c.lower():
                    protein_col = c
                    break
            if protein_col is None:
                protein_col = cols[0]
            
            # Find GO term column
            go_col = None
            for c in cols:
                if 'go' in c.lower() or 'term' in c.lower():
                    go_col = c
                    break
            if go_col is None:
                go_col = cols[1] if len(cols) > 1 else cols[0]
            
            for _, row in chunk.iterrows():
                raw_id = str(row[protein_col])
                go_term = str(row[go_col])
                
                if pd.isna(raw_id) or pd.isna(go_term):
                    continue
                
                protein_id = normalize_protein_id(raw_id)
                if go_term.startswith("GO:"):
                    protein_go_map[protein_id].append(go_term)
    
    except Exception as e:
        print(f"  Error loading GOA: {e}", flush=True)
    
    return dict(protein_go_map)


def load_goa_with_taxonomy(csv_path):
    """
    Load GOA annotations with taxonomy info.
    Returns: (protein_go_map, protein_taxon_map)
    """
    protein_go_map = defaultdict(list)
    protein_taxon_map = {}
    
    if not os.path.exists(csv_path):
        print(f"  Warning: {csv_path} not found", flush=True)
        return dict(protein_go_map), protein_taxon_map
    
    try:
        for chunk in pd.read_csv(csv_path, chunksize=100000, dtype=str):
            cols = chunk.columns.tolist()
            
            # Find relevant columns
            protein_col = None
            go_col = None
            taxon_col = None
            
            for c in cols:
                c_lower = c.lower()
                if 'protein' in c_lower or 'entry' in c_lower or 'accession' in c_lower:
                    protein_col = c
                elif 'go' in c_lower or 'term' in c_lower:
                    go_col = c
                elif 'taxon' in c_lower or 'tax' in c_lower or 'species' in c_lower:
                    taxon_col = c
            
            if protein_col is None:
                protein_col = cols[0]
            if go_col is None and len(cols) > 1:
                go_col = cols[1]
            
            for _, row in chunk.iterrows():
                raw_id = str(row[protein_col])
                protein_id = normalize_protein_id(raw_id)
                
                if go_col and go_col in row:
                    go_term = str(row[go_col])
                    if go_term.startswith("GO:"):
                        protein_go_map[protein_id].append(go_term)
                
                if taxon_col and taxon_col in row:
                    taxon = str(row[taxon_col])
                    if taxon and taxon != 'nan':
                        protein_taxon_map[protein_id] = taxon
    
    except Exception as e:
        print(f"  Error loading GOA with taxonomy: {e}", flush=True)
    
    return dict(protein_go_map), protein_taxon_map


def load_taxonomy_tsv(tsv_path):
    """
    Load taxonomy from CAFA train/test taxonomy TSV.
    Returns: dict protein_id -> taxon_id
    """
    protein_taxon_map = {}
    
    if not os.path.exists(tsv_path):
        print(f"  Warning: {tsv_path} not found", flush=True)
        return protein_taxon_map
    
    with open(tsv_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            raw_id = parts[0]
            taxon = parts[1]
            
            # Skip header
            if raw_id.lower() in ("entryid", "protein_id", "entry_id"):
                continue
            
            protein_id = normalize_protein_id(raw_id)
            protein_taxon_map[protein_id] = taxon
    
    return protein_taxon_map


def load_esm2_embeddings(embeddings_path, ids_path):
    """
    Load ESM2 embeddings and protein IDs from:
      - embeddings_path: .npy array of shape (N, D)
      - ids_path: .csv file with one column containing the protein IDs

    Returns:
        embeddings: np.ndarray
        protein_ids: list[str]
        protein_to_idx: dict[str, int]
    """
    if not os.path.exists(embeddings_path) or not os.path.exists(ids_path):
        print(f"  Warning: ESM2 embeddings not found", flush=True)
        return None, [], {}
    
    # Use memory mapping to avoid loading the full matrix into RAM at once
    embeddings = np.load(embeddings_path, mmap_mode='r')

    # Load IDs from CSV (small file, safe to read fully)
    ids_df = pd.read_csv(ids_path)
    cols = ids_df.columns.tolist()

    # Try to auto-detect the ID column
    id_col = None
    for c in cols:
        cl = c.lower()
        if 'protein' in cl or 'entry' in cl or 'id' in cl:
            id_col = c
            break
    if id_col is None:
        id_col = cols[0]

    protein_ids_raw = ids_df[id_col].tolist()

    # Normalize protein IDs to be consistent with CNN + CAFA
    protein_ids = [normalize_protein_id(str(pid)) for pid in protein_ids_raw]

    # Build dict: protein_id -> embedding index
    protein_to_idx = {pid: i for i, pid in enumerate(protein_ids)}

    print(f"  Loaded {len(protein_ids)} embeddings, dimension: {embeddings.shape[1]}", flush=True)
    return embeddings, protein_ids, protein_to_idx


def load_ppi_neighbors(pkl_path):
    """
    Load PPI neighbor data from pickle file.
    Returns: dict protein_id -> list of (neighbor_id, weight)
    """
    if not os.path.exists(pkl_path):
        print(f"  Warning: {pkl_path} not found", flush=True)
        return {}
    
    with open(pkl_path, 'rb') as f:
        ppi_data = pickle.load(f)
    
    # Normalize protein IDs in PPI data
    normalized_ppi = {}
    for protein, neighbors in ppi_data.items():
        norm_protein = normalize_protein_id(str(protein))
        norm_neighbors = [(normalize_protein_id(str(n)), w) for n, w in neighbors]
        normalized_ppi[norm_protein] = norm_neighbors
    
    return normalized_ppi


# ============================================================================
# GO GRAPH UTILITIES
# ============================================================================

def parse_go_obo(obo_path):
    """
    Parse go-basic.obo file to extract GO term relationships.
    Returns: (term_info dict, child_to_parents dict, parent_to_children dict)
    """
    term_info = {}       # go_id -> {'name': str, 'namespace': str, 'is_obsolete': bool}
    child_to_parents = defaultdict(set)  # go_id -> set of parent go_ids
    parent_to_children = defaultdict(set)  # go_id -> set of child go_ids
    
    if not os.path.exists(obo_path):
        print(f"  Warning: {obo_path} not found", flush=True)
        return term_info, dict(child_to_parents), dict(parent_to_children)
    
    current_id = None
    current_name = None
    current_namespace = None
    current_obsolete = False
    current_parents = []
    
    with open(obo_path, 'r') as f:
        for line in f:
            line = line.strip()
            
            if line == '[Term]':
                # Save previous term
                if current_id:
                    term_info[current_id] = {
                        'name': current_name,
                        'namespace': current_namespace,
                        'is_obsolete': current_obsolete
                    }
                    for parent in current_parents:
                        child_to_parents[current_id].add(parent)
                        parent_to_children[parent].add(current_id)
                
                # Reset
                current_id = None
                current_name = None
                current_namespace = None
                current_obsolete = False
                current_parents = []
            
            elif line.startswith('id: '):
                current_id = line[4:]
            
            elif line.startswith('name: '):
                current_name = line[6:]
            
            elif line.startswith('namespace: '):
                current_namespace = line[11:]
            
            elif line.startswith('is_obsolete: true'):
                current_obsolete = True
            
            elif line.startswith('is_a: '):
                # Format: is_a: GO:0008150 ! biological_process
                parts = line[6:].split(' ! ')
                parent_id = parts[0].strip()
                if parent_id.startswith('GO:'):
                    current_parents.append(parent_id)
            
            elif line.startswith('relationship: part_of '):
                # Format: relationship: part_of GO:0008150 ! biological_process
                parts = line[22:].split(' ! ')
                parent_id = parts[0].strip()
                if parent_id.startswith('GO:'):
                    current_parents.append(parent_id)
    
    # Save last term
    if current_id:
        term_info[current_id] = {
            'name': current_name,
            'namespace': current_namespace,
            'is_obsolete': current_obsolete
        }
        for parent in current_parents:
            child_to_parents[current_id].add(parent)
            parent_to_children[parent].add(current_id)
    
    return term_info, dict(child_to_parents), dict(parent_to_children)


class GOGraph:
    """Utility class for GO graph operations."""
    
    def __init__(self, child_to_parents, parent_to_children, go2idx, idx2go):
        self.child_to_parents = child_to_parents
        self.parent_to_children = parent_to_children
        self.go2idx = go2idx
        self.idx2go = idx2go
        self.num_terms = len(go2idx)
        
        # Pre-compute ancestors for each term (for faster propagation)
        self._ancestors_cache = {}
    
    def get_parents(self, go_id):
        """Get direct parents of a GO term."""
        return self.child_to_parents.get(go_id, set())
    
    def get_children(self, go_id):
        """Get direct children of a GO term."""
        return self.parent_to_children.get(go_id, set())
    
    def get_all_ancestors(self, go_id):
        """Get all ancestors (transitive closure) of a GO term."""
        if go_id in self._ancestors_cache:
            return self._ancestors_cache[go_id]
        
        ancestors = set()
        to_visit = list(self.get_parents(go_id))
        
        while to_visit:
            parent = to_visit.pop()
            if parent not in ancestors:
                ancestors.add(parent)
                to_visit.extend(self.get_parents(parent))
        
        self._ancestors_cache[go_id] = ancestors
        return ancestors
    
    def propagate_up(self, scores):
        """
        Propagate scores upward in the GO hierarchy.
        Enforces: score[parent] = max(score[parent], score[child])
        
        Args:
            scores: np.array of shape (num_terms,)
        Returns:
            np.array of shape (num_terms,) with propagated scores
        """
        propagated = scores.copy()
        
        # Process terms in order of depth (children before parents)
        # Simple approach: iterate until no changes
        changed = True
        max_iters = 50  # Safety limit
        
        for _ in range(max_iters):
            changed = False
            for child_idx in range(self.num_terms):
                child_go = self.idx2go[child_idx]
                child_score = propagated[child_idx]
                
                for parent_go in self.get_parents(child_go):
                    if parent_go in self.go2idx:
                        parent_idx = self.go2idx[parent_go]
                        if propagated[parent_idx] < child_score:
                            propagated[parent_idx] = child_score
                            changed = True
            
            if not changed:
                break
        
        return propagated
    
    def negative_propagation(self, scores, negative_mask, penalty=0.1):
        """
        Apply negative propagation: if a term is impossible, 
        ancestors also get down-weighted.
        
        Args:
            scores: np.array of shape (num_terms,)
            negative_mask: np.array of bool, True for terms with strong negative evidence
            penalty: multiplicative penalty factor
        Returns:
            np.array of shape (num_terms,) with penalized scores
        """
        penalized = scores.copy()
        
        for term_idx in range(self.num_terms):
            if negative_mask[term_idx]:
                term_go = self.idx2go[term_idx]
                # Penalize all ancestors
                for ancestor_go in self.get_all_ancestors(term_go):
                    if ancestor_go in self.go2idx:
                        anc_idx = self.go2idx[ancestor_go]
                        penalized[anc_idx] *= penalty
        
        return penalized


# ============================================================================
# GO VOCABULARY BUILDER
# ============================================================================

def build_go_vocabulary(cafa_labels, goa_labels, go_term_info, min_freq=MIN_GO_FREQ):
    """
    Build a unified GO vocabulary from CAFA + GOA labels.
    Only keep terms with frequency >= min_freq and that exist in go-basic.obo.
    
    Returns: (go2idx dict, idx2go list)
    """
    # Count GO term frequencies
    go_counter = Counter()
    
    for go_terms in cafa_labels.values():
        for go in go_terms:
            go_counter[go] += 1
    
    for go_terms in goa_labels.values():
        for go in go_terms:
            go_counter[go] += 1
    
    print(f"  Total unique GO terms in CAFA+GOA: {len(go_counter)}", flush=True)
    
    # Filter by frequency and existence in OBO
    valid_gos = set(go_term_info.keys())
    frequent_gos = [
        go for go, count in go_counter.items() 
        if count >= min_freq and go in valid_gos
    ]
    frequent_gos.sort()
    
    go2idx = {go: i for i, go in enumerate(frequent_gos)}
    idx2go = frequent_gos
    
    print(f"  GO terms with freq >= {min_freq} and in OBO: {len(frequent_gos)}", flush=True)
    
    return go2idx, idx2go


# ============================================================================
# CHANNEL 1: CNN PREDICTIONS (PRE-COMPUTED)
# ============================================================================

def load_cnn_predictions(submission_path, go2idx):
    """
    Load pre-computed CNN predictions from submission file.
    
    Returns: dict protein_id -> np.array of scores
    """
    cnn_scores = {}
    num_terms = len(go2idx)
    
    if not os.path.exists(submission_path):
        print(f"  Warning: {submission_path} not found", flush=True)
        return cnn_scores
    
    # Read submission file
    # Format: EntryID<TAB>term<TAB>score (with header)
    protein_term_scores = defaultdict(dict)
    
    with open(submission_path, 'r') as f:
        header = f.readline()  # Skip header
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            
            protein_id = normalize_protein_id(parts[0])
            go_term = parts[1]
            try:
                score = float(parts[2])
            except ValueError:
                continue
            
            protein_term_scores[protein_id][go_term] = score
    
    # Convert to numpy arrays
    for protein_id, term_scores in protein_term_scores.items():
        scores = np.zeros(num_terms, dtype=np.float16)
        for go_term, score in term_scores.items():
            if go_term in go2idx:
                scores[go2idx[go_term]] = score
        cnn_scores[protein_id] = scores
    
    print(f"  Loaded CNN predictions for {len(cnn_scores)} proteins", flush=True)
    
    return cnn_scores


# ============================================================================
# CHANNEL 2: ESM2 + MLP MODEL
# ============================================================================

class ESM2Dataset(Dataset):
    """Dataset for ESM2 embedding-based training."""
    
    def __init__(self, protein_ids, embeddings, protein_to_idx, labels, go2idx):
        self.protein_ids = protein_ids
        self.embeddings = embeddings
        self.protein_to_idx = protein_to_idx
        self.labels = labels
        self.go2idx = go2idx
        self.num_terms = len(go2idx)
    
    def __len__(self):
        return len(self.protein_ids)
    
    def __getitem__(self, idx):
        protein_id = self.protein_ids[idx]
        emb_idx = self.protein_to_idx[protein_id]
        embedding = self.embeddings[emb_idx].astype(np.float32)
        
        # Build label vector
        label_vec = np.zeros(self.num_terms, dtype=np.float32)
        for go_term in self.labels.get(protein_id, []):
            if go_term in self.go2idx:
                label_vec[self.go2idx[go_term]] = 1.0
        
        return torch.from_numpy(embedding), torch.from_numpy(label_vec)


class ESM2TestDataset(Dataset):
    """Dataset for ESM2 embedding-based inference."""
    
    def __init__(self, protein_ids, embeddings, protein_to_idx):
        self.protein_ids = protein_ids
        self.embeddings = embeddings
        self.protein_to_idx = protein_to_idx
    
    def __len__(self):
        return len(self.protein_ids)
    
    def __getitem__(self, idx):
        protein_id = self.protein_ids[idx]
        emb_idx = self.protein_to_idx[protein_id]
        embedding = self.embeddings[emb_idx].astype(np.float32)
        return torch.from_numpy(embedding), protein_id


class ESM2MLP(nn.Module):
    """Simple MLP model for GO term prediction from ESM2 embeddings."""
    
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def train_esm2_epoch(model, dataloader, criterion, optimizer, device, epoch_idx, num_epochs):
    """Train one epoch with progress logging."""
    model.train()
    total_loss = 0.0
    num_batches = len(dataloader)
    epoch_start = time.time()
    
    for batch_idx, (embeddings, labels) in enumerate(dataloader, start=1):
        batch_start = time.time()
        
        embeddings = embeddings.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        logits = model(embeddings)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        batch_time = time.time() - batch_start
        
        if batch_idx % 50 == 0 or batch_idx == num_batches:
            avg_loss = total_loss / batch_idx
            elapsed = time.time() - epoch_start
            avg_batch = elapsed / batch_idx
            eta = (num_batches - batch_idx) * avg_batch
            
            print(
                f"Epoch {epoch_idx+1}/{num_epochs} | "
                f"Batch {batch_idx}/{num_batches} | "
                f"Batch time: {batch_time:.3f}s | "
                f"Avg loss: {avg_loss:.4f} | "
                f"ETA: {format_time(eta)}",
                flush=True
            )
    
    return total_loss / num_batches


def validate_esm2_epoch(model, dataloader, criterion, device):
    """Validate one epoch."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for embeddings, labels in dataloader:
            embeddings = embeddings.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            logits = model(embeddings)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            num_batches += 1
    
    return total_loss / max(1, num_batches)


def train_esm2_model(model, train_loader, val_loader, num_epochs, device):
    """Full training loop for ESM2 MLP."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=ESM2_LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=1
    )
    
    best_val_loss = float('inf')
    best_state = None
    
    print(f"  Training ESM2 MLP for {num_epochs} epochs...", flush=True)
    print(f"  Total batches per epoch: {len(train_loader)}", flush=True)
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        
        train_loss = train_esm2_epoch(
            model, train_loader, criterion, optimizer, device, epoch, num_epochs
        )
        val_loss = validate_esm2_epoch(model, val_loader, criterion, device)
        
        scheduler.step(val_loss)
        epoch_time = time.time() - epoch_start
        
        print(
            f"  Epoch {epoch+1}/{num_epochs} - "
            f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f} "
            f"({format_time(epoch_time)})",
            flush=True
        )
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"    New best model (val_loss: {best_val_loss:.4f})", flush=True)
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model


def predict_esm2(model, dataloader, device):
    """Generate predictions for all proteins in dataloader."""
    model.eval()
    predictions = {}
    
    with torch.no_grad():
        for embeddings, protein_ids in dataloader:
            embeddings = embeddings.to(device, non_blocking=True)
            logits = model(embeddings)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            for i, pid in enumerate(protein_ids):
                # store as float16 to save RAM
                predictions[pid] = probs[i].astype(np.float16)
    
    return predictions


# ============================================================================
# CHANNEL 3: PPI NEIGHBOR PRIOR
# ============================================================================

def build_ppi_scores(ppi_neighbors, combined_labels, go2idx):
    """
    Build PPI-based scores using label propagation from neighbors.

    For each protein p with known labels, for each neighbor n of p,
    add the label vector of p to n multiplied by edge weight.
    Normalize by total edge weight per neighbor.

    This version is memory-safe: it only builds dense label vectors
    for proteins that actually appear as PPI sources, instead of
    all proteins in combined_labels.
    """
    num_terms = len(go2idx)
    ppi_scores = {}

    if not ppi_neighbors:
        print("  No PPI neighbors available, skipping PPI scores", flush=True)
        return ppi_scores

    # ------------------------------------------------------------------
    # 1) Build label vectors ONLY for proteins that are PPI sources
    #    (keys in ppi_neighbors). This avoids allocating a huge
    #    [num_proteins x num_terms] matrix.
    # ------------------------------------------------------------------
    ppi_source_ids = set(ppi_neighbors.keys())
    label_vectors = {}

    for protein_id in ppi_source_ids:
        go_terms = combined_labels.get(protein_id)
        if not go_terms:
            continue

        vec = np.zeros(num_terms, dtype=np.float32)
        for go in go_terms:
            idx = go2idx.get(go)
            if idx is not None:
                vec[idx] = 1.0

        if vec.sum() > 0:
            label_vectors[protein_id] = vec

    print(f"  PPI proteins with labels: {len(label_vectors)}", flush=True)

    # ------------------------------------------------------------------
    # 2) Aggregate neighbor labels (same logic as before, but now
    #    using the smaller label_vectors dict).
    # ------------------------------------------------------------------
    from collections import defaultdict

    neighbor_scores = defaultdict(lambda: np.zeros(num_terms, dtype=np.float32))
    neighbor_weights = defaultdict(float)

    for protein_id, neighbors in ppi_neighbors.items():
        if protein_id not in label_vectors:
            continue

        protein_labels = label_vectors[protein_id]

        for neighbor_id, weight in neighbors:
            neighbor_scores[neighbor_id] += protein_labels * weight
            neighbor_weights[neighbor_id] += weight

    # ------------------------------------------------------------------
    # 3) Normalize by total weight and compute some stats
    # ------------------------------------------------------------------
    proteins_with_neighbors = 0
    total_labeled_neighbors = 0

    labeled_sources = set(label_vectors.keys())

    for neighbor_id, scores in neighbor_scores.items():
        total_weight = neighbor_weights[neighbor_id]
        if total_weight > 0:
            ppi_scores[neighbor_id] = (scores / total_weight).astype(np.float16)
            proteins_with_neighbors += 1

            # count how many of its neighbors had labels
            labeled_count = 0
            for n_id, _ in ppi_neighbors.get(neighbor_id, []):
                if n_id in labeled_sources:
                    labeled_count += 1
            total_labeled_neighbors += labeled_count

    if proteins_with_neighbors > 0:
        mean_neighbors = total_labeled_neighbors / proteins_with_neighbors
        print(f"  Proteins with PPI scores: {len(ppi_scores)}", flush=True)
        print(
            f"  Proteins with at least one labeled neighbor: {proteins_with_neighbors}",
            flush=True,
        )
        print(
            f"  Mean number of labeled neighbors: {mean_neighbors:.2f}",
            flush=True,
        )
    else:
        print("  No PPI scores generated (no overlap with labeled proteins)", flush=True)

    return ppi_scores


# ============================================================================
# CHANNEL 4: TAXONOMY PRIOR
# ============================================================================

def build_taxonomy_priors(
    train_taxonomy, test_taxonomy, goa_taxonomy,
    combined_labels, go2idx,
    smoothing=TAXO_SMOOTHING, min_prob=TAXO_MIN_PROB, max_prob=TAXO_MAX_PROB
):
    """
    Build P(term | taxon) tables using training + GOA proteins.
    
    Returns: (taxo_scores dict, global_prior np.array)
    """
    num_terms = len(go2idx)
    
    # Combine all taxonomy info
    all_taxonomy = {}
    all_taxonomy.update(goa_taxonomy)  # GOA first (lower priority)
    all_taxonomy.update(train_taxonomy)  # CAFA train (higher priority)
    
    print(f"  Proteins with taxonomy info: {len(all_taxonomy)}", flush=True)
    
    # Count term frequencies per taxon
    taxon_term_counts = defaultdict(lambda: np.zeros(num_terms, dtype=np.float32))
    taxon_protein_counts = defaultdict(int)
    global_term_counts = np.zeros(num_terms, dtype=np.float32)
    total_proteins = 0
    
    for protein_id, go_terms in combined_labels.items():
        if protein_id not in all_taxonomy:
            continue
        
        taxon = all_taxonomy[protein_id]
        taxon_protein_counts[taxon] += 1
        total_proteins += 1
        
        for go in go_terms:
            if go in go2idx:
                idx = go2idx[go]
                taxon_term_counts[taxon][idx] += 1
                global_term_counts[idx] += 1
    
    print(f"  Distinct taxa with labeled proteins: {len(taxon_term_counts)}", flush=True)
    
    # Compute global prior (fallback)
    global_prior = (global_term_counts + smoothing) / (total_proteins + smoothing * num_terms)
    global_prior = np.clip(global_prior, min_prob, max_prob)
    
    # Compute per-taxon priors
    taxon_priors = {}
    for taxon, counts in taxon_term_counts.items():
        n_proteins = taxon_protein_counts[taxon]
        prior = (counts + smoothing) / (n_proteins + smoothing * num_terms)
        prior = np.clip(prior, min_prob, max_prob)
        taxon_priors[taxon] = prior
    
    # Build taxonomy scores for test proteins
    taxo_scores = {}
    
    # Add test proteins
    for protein_id, taxon in test_taxonomy.items():
        if taxon in taxon_priors:
            taxo_scores[protein_id] = taxon_priors[taxon].astype(np.float16)
        else:
            taxo_scores[protein_id] = global_prior.astype(np.float16)
    
    # Add any labeled proteins not in test
    for protein_id, taxon in all_taxonomy.items():
        if protein_id not in taxo_scores:
            if taxon in taxon_priors:
                taxo_scores[protein_id] = taxon_priors[taxon].astype(np.float16)
            else:
                taxo_scores[protein_id] = global_prior.astype(np.float16)
    
    # Sanity check: print top GO terms for a few taxa
    print(f"  Sample taxonomy priors:", flush=True)
    sample_taxa = list(taxon_priors.keys())[:3]
    idx2go_list = [None] * len(go2idx)
    for go, i in go2idx.items():
        idx2go_list[i] = go
    
    for taxon in sample_taxa:
        prior = taxon_priors[taxon]
        top_indices = np.argsort(prior)[-5:][::-1]
        top_terms = [(idx2go_list[i], prior[i]) for i in top_indices]
        print(f"    Taxon {taxon}: {top_terms}", flush=True)
    
    return taxo_scores, global_prior


# ============================================================================
# BLENDER
# ============================================================================

def blend_channels_for_protein(
    protein_id,
    cnn_scores, esm2_scores, ppi_scores, taxo_scores,
    num_terms,
    w_cnn=WEIGHT_CNN, w_esm2=WEIGHT_ESM2, w_ppi=WEIGHT_PPI, w_taxo=WEIGHT_TAXO
):
    """
    Blend all channels for a single protein into a score vector.
    Returns: (scores, channel_flags_dict)
    """
    channels = []
    weights = []
    flags = {'cnn': False, 'esm2': False, 'ppi': False, 'taxo': False}

    if protein_id in cnn_scores:
        channels.append(cnn_scores[protein_id])
        weights.append(w_cnn)
        flags['cnn'] = True

    if protein_id in esm2_scores:
        channels.append(esm2_scores[protein_id])
        weights.append(w_esm2)
        flags['esm2'] = True

    if protein_id in ppi_scores:
        channels.append(ppi_scores[protein_id])
        weights.append(w_ppi)
        flags['ppi'] = True

    if protein_id in taxo_scores:
        channels.append(taxo_scores[protein_id])
        weights.append(w_taxo)
        flags['taxo'] = True

    if not channels:
        scores = np.zeros(num_terms, dtype=np.float16)
        return scores, flags

    total_weight = float(sum(weights))
    weights = [w / total_weight for w in weights]

    scores = np.zeros(num_terms, dtype=np.float16)
    for ch, w in zip(channels, weights):
        scores += ch * w

    return scores, flags


# ============================================================================
# STREAMING BLENDER + SUBMISSION WRITER
# ============================================================================

def blend_and_write_submission(
    test_protein_ids,
    cnn_scores, esm2_scores, ppi_scores, taxo_scores,
    go_graph, idx2go,
    output_path,
    num_terms,
    w_cnn=WEIGHT_CNN, w_esm2=WEIGHT_ESM2, w_ppi=WEIGHT_PPI, w_taxo=WEIGHT_TAXO,
    apply_go_propagation=APPLY_GO_PROPAGATION,
    top_k=TOP_K_PER_PROTEIN,
    min_score=MIN_SCORE_EPSILON,
):
    """
    Streamed version:
      * blends channels per protein
      * optionally applies GO propagation
      * writes top-K lines immediately
      * does NOT keep all scores in memory
    """
    missing_counts = {'cnn': 0, 'esm2': 0, 'ppi': 0, 'taxo': 0}
    n_lines = 0

    with open(output_path, 'w') as f:
        # Write CAFA / Kaggle-compatible header
        f.write("EntryID\tterm\tscore\n")

        for i, protein_id in enumerate(test_protein_ids, start=1):
            scores, flags = blend_channels_for_protein(
                protein_id,
                cnn_scores, esm2_scores, ppi_scores, taxo_scores,
                num_terms,
                w_cnn=w_cnn, w_esm2=w_esm2, w_ppi=w_ppi, w_taxo=w_taxo,
            )

            # Update missing counts
            for ch in missing_counts.keys():
                if not flags[ch]:
                    missing_counts[ch] += 1

            # GO propagation stays ON (unless you flip the flag)
            if apply_go_propagation and (go_graph is not None):
                scores = go_graph.propagate_up(scores)

            # Top-K
            if len(scores) <= top_k:
                top_indices = np.argsort(scores)[::-1]
            else:
                top_indices = np.argpartition(scores, -top_k)[-top_k:]
                top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

            for idx in top_indices:
                score = float(scores[idx])
                score = max(min_score, min(1.0, score))
                go_term = idx2go[idx]
                f.write(f"{protein_id}\t{go_term}\t{score:.6f}\n")
                n_lines += 1

            if i % 10000 == 0:
                print(f"  Processed {i}/{len(test_protein_ids)} proteins...", flush=True)

    n_proteins = len(test_protein_ids)
    print(f"  Channel coverage:", flush=True)
    print(
        f"    CNN:   {n_proteins - missing_counts['cnn']}/{n_proteins} "
        f"({100*(n_proteins - missing_counts['cnn'])/n_proteins:.1f}%)",
        flush=True,
    )
    print(
        f"    ESM2:  {n_proteins - missing_counts['esm2']}/{n_proteins} "
        f"({100*(n_proteins - missing_counts['esm2'])/n_proteins:.1f}%)",
        flush=True,
    )
    print(
        f"    PPI:   {n_proteins - missing_counts['ppi']}/{n_proteins} "
        f"({100*(n_proteins - missing_counts['ppi'])/n_proteins:.1f}%)",
        flush=True,
    )
    print(
        f"    Taxo:  {n_proteins - missing_counts['taxo']}/{n_proteins} "
        f"({100*(n_proteins - missing_counts['taxo'])/n_proteins:.1f}%)",
        flush=True,
    )

    print(f"  Submission written to {output_path}", flush=True)
    print(f"  Total lines: {n_lines:,}", flush=True)


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    print("=" * 70, flush=True)
    print("CAFA 6 Multi-Channel Protein Function Prediction", flush=True)
    print("=" * 70, flush=True)
    
    total_start = time.time()
    
    # -------------------------------------------------------------------------
    # Step 1: Load GO ontology
    # -------------------------------------------------------------------------
    print("\n[1/9] Loading GO ontology...", flush=True)
    go_term_info, child_to_parents, parent_to_children = parse_go_obo(GO_OBO_PATH)
    print(f"  Loaded {len(go_term_info)} GO terms from OBO", flush=True)
    
    # -------------------------------------------------------------------------
    # Step 2: Load labels (CAFA + GOA)
    # -------------------------------------------------------------------------
    print("\n[2/9] Loading labels...", flush=True)
    
    cafa_labels = load_cafa_train_labels(CAFA_TRAIN_TERMS)
    print(f"  CAFA train labels: {len(cafa_labels)} proteins", flush=True)
    
    goa_labels = load_goa_annotations(GOA_CSV_PATH)
    print(f"  GOA annotations: {len(goa_labels)} proteins", flush=True)
    
    # Combine labels (CAFA takes precedence for overlapping proteins)
    combined_labels = {}
    combined_labels.update(goa_labels)
    for pid, terms in cafa_labels.items():
        if pid in combined_labels:
            combined_labels[pid] = list(set(combined_labels[pid] + terms))
        else:
            combined_labels[pid] = terms
    print(f"  Combined labels: {len(combined_labels)} proteins", flush=True)
    
    # -------------------------------------------------------------------------
    # Step 3: Build GO vocabulary
    # -------------------------------------------------------------------------
    print("\n[3/9] Building GO vocabulary...", flush=True)
    go2idx, idx2go = build_go_vocabulary(cafa_labels, goa_labels, go_term_info, min_freq=MIN_GO_FREQ)
    num_terms = len(go2idx)
    print(f"  Final vocabulary size: {num_terms} GO terms", flush=True)
    
    # Build GO graph utilities
    go_graph = GOGraph(child_to_parents, parent_to_children, go2idx, idx2go)
    
    # -------------------------------------------------------------------------
    # Step 4: Load ESM2 embeddings
    # -------------------------------------------------------------------------
    print("\n[4/9] Loading ESM2 embeddings...", flush=True)
    esm2_embeddings, esm2_protein_ids, esm2_protein_to_idx = load_esm2_embeddings(
        ESM2_EMBEDDINGS_PATH, ESM2_IDS_PATH
    )
    esm2_available = esm2_embeddings is not None
    
    if esm2_available:
        esm2_protein_set = set(esm2_protein_ids)
        print(f"  ESM2 embedding dimension: {esm2_embeddings.shape[1]}", flush=True)
    else:
        esm2_protein_set = set()
        print(f"  ESM2 embeddings not available", flush=True)
    
    # -------------------------------------------------------------------------
    # Step 5: Load CNN predictions (Channel 1)
    # -------------------------------------------------------------------------
    print("\n[5/9] Loading CNN predictions (Channel 1)...", flush=True)
    cnn_scores = load_cnn_predictions(CNN_SUBMISSION_PATH, go2idx)
    
    # -------------------------------------------------------------------------
    # Step 6: Train ESM2 MLP (Channel 2)
    # -------------------------------------------------------------------------
    print("\n[6/9] Training ESM2 MLP (Channel 2)...", flush=True)
    esm2_scores = {}
    
    if esm2_available:
        # Find training proteins (have labels AND have ESM2 embeddings)
        train_protein_ids = [
            pid for pid in combined_labels.keys() 
            if pid in esm2_protein_set
        ]
        print(f"  Training proteins with ESM2 embeddings: {len(train_protein_ids)}", flush=True)
        
        if len(train_protein_ids) > 100:  # Only train if we have enough data
            # Split train/val
            train_ids, val_ids = train_test_split(
                train_protein_ids, test_size=ESM2_VAL_SPLIT, random_state=42
            )
            print(f"  Train: {len(train_ids)}, Val: {len(val_ids)}", flush=True)
            
            # Create datasets
            train_dataset = ESM2Dataset(
                train_ids, esm2_embeddings, esm2_protein_to_idx, combined_labels, go2idx
            )
            val_dataset = ESM2Dataset(
                val_ids, esm2_embeddings, esm2_protein_to_idx, combined_labels, go2idx
            )
            
            train_loader = DataLoader(
                train_dataset, batch_size=ESM2_BATCH_SIZE, shuffle=True, num_workers=0
            )
            val_loader = DataLoader(
                val_dataset, batch_size=ESM2_BATCH_SIZE, shuffle=False, num_workers=0
            )
            
            # Create and train model
            esm2_dim = esm2_embeddings.shape[1]
            model = ESM2MLP(esm2_dim, ESM2_HIDDEN_DIM, num_terms, ESM2_DROPOUT).to(device)
            
            total_params = sum(p.numel() for p in model.parameters())
            print(f"  ESM2 MLP parameters: {total_params:,}", flush=True)
            
            model = train_esm2_model(model, train_loader, val_loader, ESM2_NUM_EPOCHS, device)
            
            # Generate predictions for all proteins with ESM2 embeddings
            print(f"  Generating ESM2 predictions...", flush=True)
            all_esm2_dataset = ESM2TestDataset(
                esm2_protein_ids, esm2_embeddings, esm2_protein_to_idx
            )
            all_esm2_loader = DataLoader(
                all_esm2_dataset, batch_size=ESM2_BATCH_SIZE, shuffle=False, num_workers=0
            )
            esm2_scores = predict_esm2(model, all_esm2_loader, device)
            print(f"  Generated ESM2 predictions for {len(esm2_scores)} proteins", flush=True)
            
            # Free large ESM2-related objects to save RAM
            del train_dataset, val_dataset
            del train_loader, val_loader
            del all_esm2_dataset, all_esm2_loader
            del esm2_embeddings, esm2_protein_ids, esm2_protein_to_idx
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            print(f"  Not enough training data for ESM2 MLP", flush=True)
    
    # -------------------------------------------------------------------------
    # Step 7: Build PPI scores (Channel 3)
    # -------------------------------------------------------------------------
    print("\n[7/9] Building PPI scores (Channel 3)...", flush=True)
    ppi_neighbors = load_ppi_neighbors(PPI_NEIGHBORS_PATH)
    print(f"  Loaded PPI data for {len(ppi_neighbors)} proteins", flush=True)
    
    ppi_scores = build_ppi_scores(ppi_neighbors, combined_labels, go2idx)
    
    # -------------------------------------------------------------------------
    # Step 8: Build Taxonomy priors (Channel 4)
    # -------------------------------------------------------------------------
    print("\n[8/9] Building Taxonomy priors (Channel 4)...", flush=True)
    
    # Load taxonomy info
    train_taxonomy = load_taxonomy_tsv(CAFA_TRAIN_TAXONOMY)
    print(f"  Train taxonomy: {len(train_taxonomy)} proteins", flush=True)
    
    test_taxonomy = load_taxonomy_tsv(CAFA_TEST_TAXONOMY)
    print(f"  Test taxonomy: {len(test_taxonomy)} proteins", flush=True)
    
    goa_labels_with_taxo, goa_taxonomy = load_goa_with_taxonomy(GOA_TAXONOMY_CSV_PATH)
    print(f"  GOA taxonomy: {len(goa_taxonomy)} proteins", flush=True)
    
    taxo_scores, global_prior = build_taxonomy_priors(
        train_taxonomy, test_taxonomy, goa_taxonomy, combined_labels, go2idx
    )
    print(f"  Generated taxonomy priors for {len(taxo_scores)} proteins", flush=True)

    # Optional: free large label and taxonomy structures before blending
    try:
        del cafa_labels
        del goa_labels
        del combined_labels
        del train_taxonomy
        del test_taxonomy
        del goa_labels_with_taxo
        del goa_taxonomy
    except NameError:
        pass

    gc.collect()

    # -------------------------------------------------------------------------
    # Step 9: Blend channels and write submission
    # -------------------------------------------------------------------------
    print("\n[9/9] Blending channels and writing submission...", flush=True)
    
    # Collect all test protein IDs
    # We use proteins from CNN predictions as the base set
    # (since CNN was trained on the test set)
    test_protein_ids = list(cnn_scores.keys())
    
    # Also add any proteins from other channels that might be missing
    all_protein_ids = set(test_protein_ids)
    all_protein_ids.update(esm2_scores.keys())
    all_protein_ids.update(ppi_scores.keys())
    all_protein_ids.update(taxo_scores.keys())
    
    # Filter to only proteins that have at least one channel
    test_protein_ids = []
    for pid in all_protein_ids:
        has_channel = (
            pid in cnn_scores or 
            pid in esm2_scores or 
            pid in ppi_scores or 
            pid in taxo_scores
        )
        if has_channel:
            test_protein_ids.append(pid)
    
    test_protein_ids.sort()
    print(f"  Total test proteins to predict: {len(test_protein_ids)}", flush=True)
    
    # Blend channels and write submission directly (streaming)
    blend_and_write_submission(
        test_protein_ids,
        cnn_scores, esm2_scores, ppi_scores, taxo_scores,
        go_graph, idx2go,
        OUTPUT_SUBMISSION_PATH,
        num_terms,
        w_cnn=WEIGHT_CNN, w_esm2=WEIGHT_ESM2, w_ppi=WEIGHT_PPI, w_taxo=WEIGHT_TAXO,
        apply_go_propagation=APPLY_GO_PROPAGATION,
        top_k=TOP_K_PER_PROTEIN,
        min_score=MIN_SCORE_EPSILON,
    )
    
    # -------------------------------------------------------------------------
    # Done
    # -------------------------------------------------------------------------
    total_time = time.time() - total_start
    print("\n" + "=" * 70, flush=True)
    print(f"Finished. Total time: {format_time(total_time)}", flush=True)
    print(f"Final submission written to {OUTPUT_SUBMISSION_PATH}", flush=True)
    print("=" * 70, flush=True)


if __name__ == '__main__':
    main()



Using device: cuda
CAFA 6 Multi-Channel Protein Function Prediction

[1/9] Loading GO ontology...
  Loaded 48101 GO terms from OBO

[2/9] Loading labels...
  CAFA train labels: 82404 proteins
  GOA annotations: 219224 proteins
  Combined labels: 219225 proteins

[3/9] Building GO vocabulary...
  Total unique GO terms in CAFA+GOA: 28032
  GO terms with freq >= 30 and in OBO: 10018
  Final vocabulary size: 10018 GO terms

[4/9] Loading ESM2 embeddings...
  Loaded 287001 embeddings, dimension: 1280
  ESM2 embedding dimension: 1280

[5/9] Loading CNN predictions (Channel 1)...
  Loaded CNN predictions for 224309 proteins

[6/9] Training ESM2 MLP (Channel 2)...
  Training proteins with ESM2 embeddings: 219225
  Train: 197302, Val: 21923
  ESM2 MLP parameters: 11,580,194
  Training ESM2 MLP for 8 epochs...
  Total batches per epoch: 3083
Epoch 1/8 | Batch 50/3083 | Batch time: 0.004s | Avg loss: 0.5112 | ETA: 11.8min
Epoch 1/8 | Batch 100/3083 | Batch time: 0.004s | Avg loss: 0.2971 | ETA: 9